# P2P — Submission (offline inference)

**This notebook MUST run with internet OFF** (per competition rule 5). All dependencies and files come from attached Kaggle Datasets.

**Required Kaggle Dataset attachments**:
1. `pixels-to-predictions` — the competition data.
2. `p2p-code` — your `.py` files.
3. `p2p-adapter` — the trained LoRA adapter (output of the training notebook).
4. `smolvlm-500m-instruct` — the base model weights/processor (you create this once, see notes at the bottom).

**Settings panel**:
- Accelerator: **GPU T4 x1**.
- Internet: **OFF** ← critical.


## 1. Imports & paths

In [ ]:
import sys
from pathlib import Path

CODE_DIR = '/kaggle/input/p2p-code'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Inspect what's mounted under /kaggle/input/ to confirm dataset names.
!ls /kaggle/input/


In [ ]:
# Adjust these paths to match the actual mounted dataset names.
BASE_MODEL_DIR = Path('/kaggle/input/smolvlm-500m-instruct')
ADAPTER_DIR    = Path('/kaggle/input/p2p-adapter/final_adapter')
DATA_ROOT      = Path('/kaggle/input/pixels-to-predictions')

# Auto-detect data subdir (some comps use DATA_ROOT/data, some use DATA_ROOT directly)
if (DATA_ROOT / 'train.csv').exists():
    DATA_DIR = DATA_ROOT
elif (DATA_ROOT / 'data' / 'train.csv').exists():
    DATA_DIR = DATA_ROOT / 'data'
else:
    raise FileNotFoundError(f'test.csv not found under {DATA_ROOT}')

# Verify everything exists before loading anything heavy.
assert BASE_MODEL_DIR.exists(), f'Base model dir not found: {BASE_MODEL_DIR}'
assert (BASE_MODEL_DIR / 'config.json').exists(), f'config.json missing in {BASE_MODEL_DIR}'
assert ADAPTER_DIR.exists(), f'Adapter dir not found: {ADAPTER_DIR}'
assert (ADAPTER_DIR / 'adapter_config.json').exists(), f'adapter_config.json missing in {ADAPTER_DIR}'
assert (DATA_DIR / 'test.csv').exists(), f'test.csv missing in {DATA_DIR}'

print(f'BASE_MODEL_DIR : {BASE_MODEL_DIR}')
print(f'ADAPTER_DIR    : {ADAPTER_DIR}')
print(f'DATA_DIR       : {DATA_DIR}')


## 2. Disable network (defensive)

Even with the Kaggle Internet toggle off, set offline env vars so HF doesn't try to phone home and fail noisily.


In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'


## 3. Run the submission script

We import `make_submission` and call its `main()` directly so we get tracebacks in-notebook (vs. a `!python` shell call which would swallow them).


In [ ]:
import importlib, make_submission
importlib.reload(make_submission)

import sys as _sys
_old_argv = _sys.argv
_sys.argv = [
    'make_submission',
    '--base_model_path', str(BASE_MODEL_DIR),
    '--adapter_path',    str(ADAPTER_DIR),
    '--data_dir',        str(DATA_DIR),
    '--out',             '/kaggle/working/submission.csv',
    '--batch_size',      '4',
    # Add '--use_4bit' if you hit OOM. Default is fp16/bf16 (faster).
]
try:
    make_submission.main()
finally:
    _sys.argv = _old_argv


## 4. Verify the submission file


In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print(f'Rows: {len(sub)}')
print(f'Columns: {list(sub.columns)}')
print(f'answer dtype: {sub["answer"].dtype}')
print(f'answer value counts:')
print(sub['answer'].value_counts().sort_index())
print()
print('Head:')
print(sub.head())

# Sanity checks before leaderboard submission:
assert sub.columns.tolist() == ['id', 'answer'], 'columns must be exactly id,answer'
assert sub['answer'].dtype.kind == 'i', 'answer must be integer'
assert sub['id'].is_unique, 'duplicate ids in submission'
assert sub['answer'].notna().all(), 'NaN in answer column'
print('\nAll sanity checks passed. submission.csv is ready to submit.')


---

## Setup notes (read once when first creating this notebook)

### Creating the `smolvlm-500m-instruct` Kaggle Dataset

Do this once, in a **separate** notebook with internet ON:

```python
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='HuggingFaceTB/SmolVLM-500M-Instruct',
    local_dir='/kaggle/working/smolvlm-500m-instruct',
    local_dir_use_symlinks=False,
    ignore_patterns=['*.bin', 'onnx/*'],  # safetensors-only, no ONNX (saves ~3GB)
)
```

Then *Save Version → Save & Run All*, and from *Notebook → Output* publish `/kaggle/working/smolvlm-500m-instruct` as a new Dataset. Reuse it across submission runs.

### Creating the `p2p-code` Kaggle Dataset

Either upload the `.py` files directly through Kaggle's *Datasets → New Dataset* UI, or push them to a GitHub repo and use Kaggle's GitHub-import option. Update the dataset version whenever you change code.

### Creating the `p2p-adapter` Kaggle Dataset

After the training notebook completes, go to *Notebook → Output → New Dataset* and publish the `final_adapter/` directory.
